# Deep Learning 020 — Batch, Stochastic & Mini-Batch Gradient Descent

Companion notebook to the lesson. All three are the same algorithm. The only thing that
changes is **how many examples you look at before taking a step**, and that single number
controls the update count, the smoothness of the loss curve, the memory needed, and the
wall-clock time per epoch.

| Variant | Batch size | Updates per epoch (N = 4000) |
|---|---|---|
| Batch GD | N | 1 |
| Mini-batch GD | 32 | 125 |
| SGD | 1 | 4000 |

We measure all four consequences on the same problem.

In [ ]:
import time
import numpy as np

rng = np.random.default_rng(0)

N, D = 4000, 20
X = rng.normal(size=(N, D))
true_w = rng.normal(size=D)
y = (X @ true_w + rng.normal(scale=0.5, size=N) > 0).astype(float)

def sigmoid(z):  return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

def loss(w, b, Xs=X, ys=y):
    p = sigmoid(Xs @ w + b)
    return float(-(ys * np.log(p + 1e-12) + (1 - ys) * np.log(1 - p + 1e-12)).mean())

def grad(w, b, Xs, ys):
    d = (sigmoid(Xs @ w + b) - ys) / len(ys)
    return Xs.T @ d, d.sum()

print(f"{N} examples, {D} features, class balance {y.mean():.3f}")

## One training function, one parameter

`batch_size = N` is batch gradient descent. `batch_size = 1` is stochastic. Anything in
between is mini-batch. **There is no other difference** — same gradient formula, same
update rule.

In [ ]:
def train(batch_size, epochs=30, lr=0.5, seed=0):
    r = np.random.default_rng(seed)
    w, b = np.zeros(D), 0.0
    curve, updates = [loss(w, b)], 0
    t0 = time.perf_counter()
    for _ in range(epochs):
        order = r.permutation(N)                      # shuffle every epoch
        for start in range(0, N, batch_size):
            idx = order[start:start + batch_size]      # the last batch takes the remainder
            gw, gb = grad(w, b, X[idx], y[idx])
            w -= lr * gw
            b -= lr * gb
            updates += 1
            curve.append(loss(w, b))
    return dict(w=w, b=b, curve=np.array(curve), updates=updates,
                secs=time.perf_counter() - t0, final=loss(w, b))

print(f"{'variant':>12}{'batch':>8}{'updates':>10}{'final loss':>13}{'seconds':>10}")
runs = {}
for name, bs in (("batch", N), ("mini-batch", 32), ("SGD", 1)):
    r_ = train(bs)
    runs[name] = r_
    print(f"{name:>12}{bs:>8}{r_['updates']:>10}{r_['final']:>13.5f}{r_['secs']:>10.2f}")

Three things to read off, and the third is the one people get wrong.

1. **Updates per epoch** — batch takes 1, SGD takes 4000. For the *same number of epochs*
   SGD has done four thousand times as much learning.
2. **Final loss after 30 epochs** — batch GD has barely moved. It is not that batch GD is a
   worse algorithm; it has taken 30 steps in total.
3. **Wall clock** — SGD is *slower per epoch* despite touching exactly the same data, and
   the reason is not the arithmetic.

## Why SGD is slower per epoch when it does the same arithmetic

Batch GD computes its gradient as one matrix multiply over all 4000 rows. SGD does 4000
separate tiny ones. Same number of multiply-adds, wildly different efficiency — this is
**vectorisation**, and it is the whole argument for mini-batching.

In [ ]:
def timed(fn, reps=5):
    fn()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return min(ts)

w0, b0 = np.zeros(D), 0.0
def one_big():
    grad(w0, b0, X, y)

def many_small(bs):
    def f():
        for s in range(0, N, bs):
            grad(w0, b0, X[s:s + bs], y[s:s + bs])
    return f

print(f"{'batch size':>12}{'ms per epoch of gradients':>28}{'vs one matmul':>16}")
base = timed(one_big)
for bs in (N, 512, 128, 32, 8, 1):
    t = timed(many_small(bs))
    print(f"{bs:>12}{t * 1e3:>28.2f}{t / base:>15.0f}x")

The arithmetic is identical in every row. The cost is per-call overhead — Python loop, array
slicing, BLAS setup — paid once per batch. **Mini-batching exists to amortise it**, and the
curve flattens out well before you reach the full dataset, which is why 32 or 64 is a
sensible default and N is not required.

## The loss curves — noise is the visible difference

Batch GD uses the exact gradient of the whole dataset, so the loss falls monotonically. SGD
estimates it from one example, so it is right on average and wrong every single time.

In [ ]:
# resample each curve onto a per-epoch grid so the three are comparable
print(f"{'epoch':>7}{'batch':>12}{'mini-batch':>13}{'SGD':>12}")
for e in (0, 1, 2, 5, 10, 20, 30):
    row = []
    for name in ("batch", "mini-batch", "SGD"):
        c = runs[name]["curve"]
        idx = min(int(e * (len(c) - 1) / 30), len(c) - 1)
        row.append(c[idx])
    print(f"{e:>7}{row[0]:>12.5f}{row[1]:>13.5f}{row[2]:>12.5f}")

for name in ("batch", "mini-batch", "SGD"):
    c = runs[name]["curve"]
    rises = int((np.diff(c) > 0).sum())
    print(f"\n{name:>11}: {len(c) - 1} steps, loss went UP on {rises} of them"
          f" ({rises / max(len(c) - 1, 1):.1%})")

Batch GD never goes uphill — 0 of 30 steps. SGD goes uphill on **51.7%** of its steps,
barely better than a coin flip, and still finishes lower than batch GD (0.170 against
0.240) because it took four thousand times as many. Mini-batch beats both at **0.079**,
which is the practical answer: enough steps to make progress, enough examples per step to
keep the direction roughly right and the hardware busy.

**Is the noise useful?** Yes, with a caveat, and both halves are measurable. The noise lets
the parameters jump out of a shallow local minimum that the exact gradient would sit in
forever — but the same noise means it never settles exactly at the bottom.

In [ ]:
# a 1-D loss with a shallow local minimum and a deeper global one
def f(x):     return 0.06 * x ** 4 - 0.6 * x ** 3 + 0.9 * x ** 2 + 1.2 * x
def df(x):    return 0.24 * x ** 3 - 1.8 * x ** 2 + 1.8 * x + 1.2

xs = np.linspace(-3, 9, 4001)
minima = xs[1:-1][(f(xs)[1:-1] < f(xs)[:-2]) & (f(xs)[1:-1] < f(xs)[2:])]
print("local minima at x =", np.round(minima, 3), " values", np.round(f(minima), 3))

def descend(noise, start=-1.0, lr=0.03, steps=4000, seed=0):
    r = np.random.default_rng(seed)
    x = start
    for _ in range(steps):
        x -= lr * (df(x) + noise * r.normal())
    return x

print()
for noise in (0.0, 2.0, 8.0):
    ends = [round(descend(noise, seed=s), 3) for s in range(6)]
    print(f"noise {noise:>4.1f}: final x over 6 seeds {ends}")

Read the three rows carefully, because they say something more specific than "noise
helps".

- **No noise**: every seed ends at −0.451, the shallow minimum worth −0.30. The exact
  gradient is trapped, and running longer changes nothing.
- **Noise 2.0**: still trapped. It jitters around the same basin — enough noise to be
  imprecise, not enough to escape. **The worst of both.**
- **Noise 8.0**: every seed escapes to the deep basin near 6.147, worth −12.31. But look
  at the spread — 5.72 to 6.54. It found the right valley and cannot sit still in it.

That is the trade: **the noise is what finds the better basin, and the same noise is what
stops you landing cleanly in it — and a middling amount buys neither.** In practice you get
both by using mini-batches (some noise) and decaying the learning rate (less noise later),
which is exactly what the next lessons on optimisers are about.

## The last batch takes the remainder

A detail that trips people up when the batch size does not divide the dataset.

In [ ]:
for n, bs in ((400, 150), (4000, 32), (1000, 256)):
    sizes = [len(range(s, min(s + bs, n))) for s in range(0, n, bs)]
    print(f"N = {n:>5}, batch = {bs:>4} -> {len(sizes)} batches, sizes "
          f"{sizes[:3]}{' ... ' if len(sizes) > 4 else ' '}{sizes[-1]}")

And powers of two — 32, 64, 128, 256 — are a **convention**, not a rule. They line up
tidily with memory hardware. Nothing breaks with `batch_size = 100`; the timing table above
would look much the same.

## Summary

| | Batch | Mini-batch | SGD |
|---|---|---|---|
| batch size | N | 32–256 | 1 |
| updates per epoch | 1 | N/B | N |
| gradient | exact | noisy estimate | very noisy estimate |
| loss curve | monotone | slightly bumpy | spiky |
| vectorised | fully | mostly | not at all |
| memory | whole dataset | one batch | one row |
| escapes shallow minima | no | somewhat | yes |

**In Keras this is one argument**: `model.fit(..., batch_size=N)` for batch,
`batch_size=1` for SGD, `batch_size=32` — the default — for mini-batch.

## Try it yourself

1. Run `train(32)` with `epochs=1` and `train(N)` with `epochs=125`. Both take 125 updates.
   Which ends lower, and why is that not a fair comparison in wall-clock terms?
2. Add learning-rate decay to `train` (`lr / (1 + 0.01 * update)`) and re-measure the "loss
   went up" percentage for SGD. Does it fall?
3. Sweep `batch_size` over `[1, 4, 16, 64, 256, 1024, N]` and plot final loss against
   seconds. Where is the knee?
4. In the noise experiment, start at `x = 6` instead of `-1`. Does noise still help, or does
   it now push you *out* of the good basin?